## Georgia Update SeaWulf Data with EI Analysis

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

import dill

In [ ]:
demographics = ['white', 'black', 'latino', 'other']
candidates = ['harris', 'trump']

### Import Models

In [ ]:
path = '../output/Georgia/models/ei_models.pkl'

In [ ]:
with open(path, 'rb') as f:
    models = dill.load(f)

In [ ]:
type(models)

#### Get Sampled Shares Per Model 

In [ ]:
def get_sample_shares(models):
    df = pd.DataFrame()

    for candidate in candidates:
        for demographic in demographics:
            ei = models[candidate][demographic]

            samples_matrix = ei.sim_trace["posterior"]["b_1"].stack(all_draws=["chain", "draw"]).values
            
            precinct_means = samples_matrix.mean(axis=1) 
            
            df[f'{demographic}_{candidate}_pct'] = precinct_means
    
    return df

In [ ]:
sampled_shares = get_sample_shares(models)
sampled_shares

### Update Seawulf Data

In [ ]:
import geopandas
gdf = geopandas.read_file("../output/Georgia/ga_seawulf.gpkg", driver="GPKG")
gdf.to_csv("../output/Georgia/ga_seawulf.csv", index=False)
gdf

In [ ]:
gdf = pd.concat([gdf, sampled_shares], axis=1)
gdf.columns = gdf.columns.str.capitalize()
gdf

In [ ]:
gdf.to_file("../output/Georgia/ga_seawulf.gpkg", driver="GPKG")

In [ ]:
gdf = geopandas.read_file("../output/Georgia/ga_seawulf.gpkg", driver="GPKG")
gdf.to_csv("../output/Georgia/ga_seawulf.csv", index=False)
gdf

In [ ]:
gdf.columns